In [17]:
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
import os

In [18]:
# Hide GPU from TensorFlow to avoid CUDA_ERROR_INVALID_HANDLE
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

In [19]:
import tensorflow as tf
from tensorflow import keras

# Verify it's using the CPU
print("Physical Devices:", tf.config.list_physical_devices())

Physical Devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


In [20]:
train_dir = "../datafiles/cats_and_dogs_small/train"
validation_dir = "../datafiles/cats_and_dogs_small/validation"
test_dir = "../datafiles/cats_and_dogs_small/test"

## Data Preprocessing
- Read all picture files.
- Decode the JPEG content to RGB grids of pixels.
- Convert these into floating point tensors.
- Rescale the pixel values (between 0 and 255) to the [0, 1] interval.

In [ ]:
# Generating batches of tensor image data with real-time data augmentation. The data will be looped over (in batches) indefinitely.
train_datagen = keras.preprocessing.image.ImageDataGenerator(rescale=1.0 / 255)
validation_datagen = keras.preprocessing.image.ImageDataGenerator(rescale=1.0 / 255)

train_generator = keras.utils.image_dataset_from_directory(
    train_dir,
    labels="inferred",
    label_mode="int",
    class_names=None,
    color_mode="rgb",
    batch_size=20,
    image_size=(150, 150),
    shuffle=True,
    seed=None,
    validation_split=None,
    subset=None,
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False,
)

validation_generator = keras.utils.image_dataset_from_directory(
    validation_dir,
    labels="inferred",
    label_mode="int",
    class_names=None,
    color_mode="rgb",
    batch_size=20,
    image_size=(150, 150),
    shuffle=True,
    seed=None,
    validation_split=None,
    subset=None,
    interpolation="bilinear",
    follow_links=False,
    crop_to_aspect_ratio=False,
)


Link: https://keras.io/preprocessing/image/

In [ ]:
import keras
from keras import layers, models

In [ ]:
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation="relu", input_shape=(150, 150, 3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation="relu"))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation="relu"))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation="relu"))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Flatten())
model.add(layers.Dense(512, activation="relu"))
model.add(layers.Dense(1, activation="sigmoid"))

In [ ]:
model.summary()

In [ ]:
from keras import optimizers

model.compile(
    loss="binary_crossentropy",
    optimizer=optimizers.RMSprop(learning_rate=1e-4),
    metrics=["accuracy"],
)

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=100,
    epochs=20,
    validation_data=validation_generator,
    validation_steps=50
)


In [ ]:
pd.DataFrame(history.history).plot(figsize=(8, 5))
plt.grid(True)
plt.gca().set_ylim(0, 1)
plt.show()

In [ ]:
model.save("cats_and_dogs_small_1.keras")

In [ ]:
del model
# clear the Keras session to free up resources after saving the model
keras.backend.clear_session()

In [ ]:
# use imported layers
data_augmentation = keras.Sequential([
    layers.Rescaling(1./255),                    # replaces rescale=1./255
    layers.RandomFlip("horizontal"),             # horizontal_flip=True
    layers.RandomRotation(40/360),               # rotation_range=40  → factor of 2π
    layers.RandomTranslation(height_factor=0.2, width_factor=0.2),  # shift_range=0.2
    layers.RandomZoom(0.2),
    # fill_mode='nearest' is the default in most of these layers
])

train_generator = keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(150, 150),
    batch_size=32,
    label_mode="binary",
).map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

validation_generator = keras.utils.image_dataset_from_directory(
    validation_dir,
    image_size=(150, 150),
    batch_size=32,
    label_mode="binary",
)

In [13]:
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation="relu", input_shape=(150, 150, 3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation="relu"))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation="relu"))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(128, (3, 3), activation="relu"))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Flatten())
model.add(layers.Dropout(0.5))  # add dropout layer to reduce overfitting
model.add(layers.Dense(512, activation="relu"))
model.add(layers.Dense(1, activation="sigmoid"))

model.compile(
    loss="binary_crossentropy",
    optimizer=optimizers.RMSprop(learning_rate=1e-4),
    metrics=["accuracy"],
)

In [22]:
history = model.fit(
    train_generator,
    steps_per_epoch=50,
    epochs=10,
    validation_data=validation_generator,
    validation_steps=50
)

Epoch 1/10
50/50 [==============================] - 55s 1s/step - loss: 0.5489 - accuracy: 0.7169
Epoch 2/10
50/50 [==============================] - 12s 224ms/step - loss: 0.5174 - accuracy: 0.7350


In [23]:
import tensorflow as tf

# Assuming train_generator and validation_generator are already defined

# Use repeat() to create a dataset that can be repeated
train_generator = tf.data.Dataset.from_generator(
    train_generator,  # Your generator function
    output_signature=train_generator.output_signature  # Important!
)

# Repeat the dataset to cover the required number of batches
train_generator = train_generator.repeat(epochs=10)  # Repeat for 10 epochs

validation_generator = tf.data.Dataset.from_generator(
    validation_generator,
    output_signature=validation_generator.output_signature
)

validation_generator = validation_generator.repeat(epochs=10)

# Now, your training looks like this:
history = model.fit(
    train_generator,
    steps_per_epoch=50,
    epochs=10,
    validation_data=validation_generator,
    validation_steps=50
)


AttributeError: 'PrefetchDataset' object has no attribute 'output_signature'

In [19]:
model.save("cats_and_dogs_small_2_py.keras")

In [20]:
pd.DataFrame(history.history).plot(figsize=(8, 5))
plt.grid(True)
plt.gca().set_ylim(0, 1)
plt.show()

ValueError: All arrays must be of the same length